# Cross-Network Prognosis Leaderboard

Walks all `checkpoints_prognoser_{combo}/` directories, parses `run_summary.json`,
and ranks (network_combo × method) combinations by test C-index, IBS, and time-dependent AUC.

Run after sweeping `PROGNOSER_RUNNER.ipynb` over multiple combos × methods.

In [ ]:
import sys, json
from pathlib import Path

REPO_ROOT = Path('/mnt/e/fyassine/ad-early-detection')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
PROGNOSER_DIR = REPO_ROOT / 'PROGNOSER' / 'notebooks'
ARTIFACTS = PROGNOSER_DIR / '_artifacts_'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

In [ ]:
def collect_runs(prognoser_dir: Path) -> pd.DataFrame:
    """Walk checkpoints_prognoser_*/ subdirs, parse run_summary.json, return tidy df."""
    rows = []
    for combo_dir in sorted(prognoser_dir.glob('checkpoints_prognoser_*')):
        if not combo_dir.is_dir():
            continue
        combo = combo_dir.name.replace('checkpoints_prognoser_', '')
        for run_dir in sorted(combo_dir.iterdir()):
            summary = run_dir / 'run_summary.json'
            if not summary.exists():
                continue
            with open(summary) as f:
                s = json.load(f)
            metrics = s.get('metrics', {})
            test = metrics.get('test', {})
            val = metrics.get('val', {})
            train = metrics.get('train', {})
            test_auc = test.get('auc', {})
            rows.append({
                'network_combo': combo,
                'method': s.get('method', 'unknown'),
                'feature_set': s.get('feature_set', ''),
                'n_features': s.get('n_features'),
                'n_train': s.get('n_train'),
                'n_test': s.get('n_test'),
                'n_events_test': s.get('n_events_test'),
                'c_train': train.get('c_index'),
                'c_val': val.get('c_index'),
                'c_test': test.get('c_index'),
                'ibs_test': test.get('ibs'),
                'auc_test_24': test_auc.get('24'),
                'auc_test_36': test_auc.get('36'),
                'auc_test_60': test_auc.get('60'),
                'run_name': s.get('run_name', run_dir.name),
                'timestamp': s.get('timestamp', ''),
            })
    return pd.DataFrame(rows)

leaderboard = collect_runs(PROGNOSER_DIR)
if leaderboard.empty:
    print('No runs found. Run PROGNOSER_RUNNER.ipynb first.')
else:
    leaderboard = leaderboard.sort_values(['c_test', 'auc_test_36'], ascending=False).reset_index(drop=True)
    print(f'Total runs: {len(leaderboard)}')
    print(f'Combos: {sorted(leaderboard.network_combo.unique())}')
    print(f'Methods: {sorted(leaderboard.method.unique())}')
leaderboard

In [ ]:
if not leaderboard.empty:
    # Best run per (combo, method) by test C-index
    best_per_combo_method = leaderboard.loc[
        leaderboard.groupby(['network_combo', 'method'])['c_test'].idxmax()
    ].reset_index(drop=True)
    print('Best run per (combo, method):')
    display_cols = ['network_combo', 'method', 'feature_set', 'n_features',
                    'c_test', 'auc_test_24', 'auc_test_36', 'ibs_test']
    print(best_per_combo_method[display_cols].to_string(index=False))

    # Save
    leaderboard.to_csv(ARTIFACTS / 'leaderboard_all_runs.csv', index=False)
    best_per_combo_method.to_csv(ARTIFACTS / 'leaderboard_best_per_combo_method.csv', index=False)

In [ ]:
if not leaderboard.empty:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)

    # Test C-index by combo and method
    pivot_c = best_per_combo_method.pivot_table(
        index='network_combo', columns='method', values='c_test'
    )
    pivot_c.plot(kind='bar', ax=axes[0], colormap='viridis', edgecolor='black', alpha=0.85)
    axes[0].set_title('Test C-index by network combo × method')
    axes[0].set_ylabel('Test C-index')
    axes[0].set_xlabel('Network combo')
    axes[0].axhline(0.5, color='red', linestyle=':', alpha=0.6, label='chance')
    axes[0].set_ylim(0, 1)
    axes[0].legend(loc='best')
    axes[0].tick_params(axis='x', rotation=30)

    # AUC@36 by combo and method
    pivot_a = best_per_combo_method.pivot_table(
        index='network_combo', columns='method', values='auc_test_36'
    )
    pivot_a.plot(kind='bar', ax=axes[1], colormap='plasma', edgecolor='black', alpha=0.85)
    axes[1].set_title('Test AUC@36 months by network combo × method')
    axes[1].set_ylabel('Time-dependent AUC at 36 months')
    axes[1].set_xlabel('Network combo')
    axes[1].axhline(0.5, color='red', linestyle=':', alpha=0.6, label='chance')
    axes[1].set_ylim(0, 1)
    axes[1].legend(loc='best')
    axes[1].tick_params(axis='x', rotation=30)

    plt.savefig(ARTIFACTS / 'leaderboard_barplot.png', dpi=150)
    plt.show()

In [ ]:
if not leaderboard.empty:
    winner = leaderboard.iloc[0]
    print('=' * 60)
    print('  TOP RUN BY TEST C-INDEX')
    print('=' * 60)
    for k in ['network_combo', 'method', 'feature_set', 'n_features',
              'c_train', 'c_val', 'c_test',
              'ibs_test', 'auc_test_24', 'auc_test_36', 'auc_test_60', 'run_name']:
        v = winner.get(k)
        if isinstance(v, float):
            print(f'  {k:>16}: {v:.4f}')
        else:
            print(f'  {k:>16}: {v}')